# ICCD753 Recuperación de Información - Examen Final
**Tema:** Diseño e Implementación de un Sistema de Recuperación de Información (RAG)
**Estudiante:** [Nombre del Estudiante]

---
### Enlace de Despliegue en la Nube
**URL Streamlit Cloud:** [https://share.streamlit.io/...](https://share.streamlit.io/...)
---


## A. Preparación del corpus
Descargamos y estructuramos la base de datos `arXiv Paper Abstracts` usando Kaggle y Pandas.

In [ ]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

print(f"Total papers: {len(df)}")
df.head(2)

## B. Representación mediante embeddings
Inicializamos el modelo de Google Gemini para obtener representaciones vectoriales del texto de los resúmenes.

In [ ]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()
embedder = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
test_embedding = embedder.embed_query("Graph Neural Networks")
print(f"Dimensión del embedding: {len(test_embedding)}")

## C. Almacenamiento y búsqueda vectorial
Se utiliza `ChromaDB` para almacenar de manera persistente los documentos y sus correspondientes vectores semánticos.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="data/chroma_db")
collection = client.get_or_create_collection(
    name="arxiv_corpus",
    metadata={"hnsw:space": "cosine"}
)
print(f"Documentos indexados actualmente: {collection.count()}")

## D. Recuperación
La clase `TextRetriever` encapsula la lógica para transformar la consulta a vector y extraer el Top-K desde ChromaDB.

In [ ]:
from src.retrieval import TextRetriever

retriever = TextRetriever()
results = retriever.retrieve("What are the main applications of Graph Neural Networks?", top_k=3)
for r in results:
    print(f"Score: {r['score']:.4f} | Title: {r['title']}")

## E. Generación aumentada por recuperación
A través de LangChain y Gemini se configura un *Prompt Estricto* para evitar alucinaciones y obligar al modelo a usar exclusivamente el contexto.

In [ ]:
from src.generation import RAGGenerator

generator = RAGGenerator()
answer = generator.generate_response(
    "What are the main applications of Graph Neural Networks?", 
    results
)
print("Respuesta Generada:\n", answer)

## F. Presentación de evidencias e Interfaz (G y H)
Toda esta lógica fue abstraída y construida dentro del script `src/app.py`. Para ejecutar la interfaz de Chat, se debe ejecutar:
```bash
streamlit run src/app.py
```
El sistema muestra debajo de cada respuesta un bloque colapsable (`st.expander`) con el título y resumen original del artículo (evidencia).